## Loading data

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
from skfp.datasets.moleculenet import load_tox21

df = load_tox21(as_frame=True)
print(df.head())

                                              SMILES  NR-AR  NR-AR-LBD  \
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1    0.0        0.0   
1                          CCN1C(=O)NC(c2ccccc2)C1=O    0.0        0.0   
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...    NaN        NaN   
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C    0.0        0.0   
4                          CC(O)(P(=O)(O)O)P(=O)(O)O    0.0        0.0   

   NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  SR-ARE  SR-ATAD5  \
0     1.0           NaN    NaN        0.0            0.0     1.0       0.0   
1     0.0           0.0    0.0        0.0            0.0     NaN       0.0   
2     NaN           NaN    NaN        NaN            NaN     0.0       NaN   
3     0.0           0.0    0.0        0.0            0.0     NaN       0.0   
4     0.0           0.0    0.0        0.0            0.0     0.0       0.0   

   SR-HSE  SR-MMP  SR-p53  
0     0.0     0.0     0.0  
1     NaN     0.0     0.0  
2 

In [3]:
print("Percentage of NaNs in each label")
print(df.drop(columns=['SMILES']).isna().mean() * 100.0)

Percentage of NaNs in each label
NR-AR             7.227685
NR-AR-LBD        13.701954
NR-AhR           16.370834
NR-Aromatase     25.667220
NR-ER            20.916869
NR-ER-LBD        11.186311
NR-PPAR-gamma    17.635040
SR-ARE           25.526753
SR-ATAD5          9.692249
SR-HSE           17.417954
SR-MMP           25.807687
SR-p53           13.497638
dtype: float64


In [4]:
df = df.dropna().reset_index(drop=True)

smiles = df['SMILES']
toxicity_labels = df.drop(columns=['SMILES'])
print(toxicity_labels)


      NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
1       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
3       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
4       0.0        0.0     0.0           0.0    0.0        0.0            1.0   
...     ...        ...     ...           ...    ...        ...            ...   
3074    0.0        0.0     0.0           0.0    1.0        0.0            0.0   
3075    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
3076    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
3077    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
3078    1.0        1.0     0.0           0.0    1.0        1.0            0.0   

      SR-ARE  SR-ATAD5  SR-

## Converting SMILES to graph adj. list representation

In [5]:
from rdkit import Chem
import torch

def smile_to_graph(smile):
    mol = Chem.MolFromSmiles(smile)
    
    node_features = []
    for atom in mol.GetAtoms():
        node_features.append([atom.GetAtomicNum()])
    
    node_features = torch.tensor(node_features, dtype=torch.float)
    
    edge_index = []
    edge_attr = []
    
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        
        bond_type = bond.GetBondTypeAsDouble()
        aromatic = 1.0 if bond.GetIsAromatic() else 0.0
        
        edge_index.append([i, j])
        edge_index.append([j, i])
        
        edge_attr.append([bond_type, aromatic])
        edge_attr.append([bond_type, aromatic])
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    return node_features, edge_index, edge_attr

# Example usage with the first SMILES in the dataset
node_features, edge_index, edge_attr = smile_to_graph(smiles[1])

print("Node features:", node_features)
print("Edge index:", edge_index)
print("Edge attributes:", edge_attr)


Node features: tensor([[ 8.],
        [16.],
        [ 8.],
        [17.],
        [ 6.],
        [ 6.],
        [ 6.],
        [ 6.],
        [ 6.],
        [ 6.]])
Edge index: tensor([[0, 1, 1, 2, 1, 3, 1, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 4],
        [1, 0, 2, 1, 3, 1, 4, 1, 5, 4, 6, 5, 7, 6, 8, 7, 9, 8, 4, 9]])
Edge attributes: tensor([[2.0000, 0.0000],
        [2.0000, 0.0000],
        [2.0000, 0.0000],
        [2.0000, 0.0000],
        [1.0000, 0.0000],
        [1.0000, 0.0000],
        [1.0000, 0.0000],
        [1.0000, 0.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000]])


In [6]:
from torch_geometric.data import Data

def prepare_graph_data(smiles, toxicity_labels):
    graph_data = []
    
    for i, smi in enumerate(smiles):
        node_features, edge_index, edge_attr = smile_to_graph(smi)
        labels = toxicity_labels.iloc[i].values
        
        # PyTorch Geometric data object
        data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor(labels, dtype=torch.float))
        graph_data.append(data)
    
    return graph_data

graph_data = prepare_graph_data(smiles, toxicity_labels)

## GINE Model

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric import nn as pyg_nn
from torch_geometric.nn import GINEConv
from torch_geometric.data import DataLoader

class GINEToxicityModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, edge_dim):
        super(GINEToxicityModel, self).__init__()
        
        self.conv1 = GINEConv(
            nn=nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)),
            edge_dim=edge_dim
        )
        
        self.conv2 = GINEConv(
            nn=nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)),
            edge_dim=edge_dim
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = self.conv1(x, edge_index, edge_attr)
        x = self.conv2(x, edge_index, edge_attr)
        
        # Graph-level pooling
        x = pyg_nn.global_mean_pool(x, data.batch)
        x = self.fc(x)
        return x


In [8]:
train_loader = DataLoader(graph_data, batch_size=32, shuffle=True)
model = GINEToxicityModel(input_dim=1, hidden_dim=64, output_dim=toxicity_labels.shape[1], edge_dim=2)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    model.train()
    print(f"Epoch {epoch+1}...", end="")
    for data in train_loader:
        optimizer.zero_grad()
        data.y = data.y.view(-1, 12)
        output = model(data)
        loss = criterion(output, data.y)
        loss.backward()
        optimizer.step()
    
    print(f" Loss: {loss.item()}")

/home/outsomniac/dev/toxicity/.venv/lib/python3.13/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1... Loss: 0.12803976237773895
Epoch 2... Loss: 0.3405001163482666
Epoch 3... Loss: 0.06796739995479584
Epoch 4... Loss: 0.028759244829416275
Epoch 5... Loss: 0.0331062413752079
Epoch 6... Loss: 0.11183694750070572
Epoch 7... Loss: 0.2843659818172455
Epoch 8... Loss: 0.10353508591651917
Epoch 9... Loss: 0.06101218983530998
Epoch 10... Loss: 0.2551070749759674
Epoch 11... Loss: 0.2092447280883789
Epoch 12... Loss: 0.02883116900920868
Epoch 13... Loss: 0.0969928577542305
Epoch 14... Loss: 0.21140722930431366
Epoch 15... Loss: 0.1770779937505722
Epoch 16... Loss: 0.0828825831413269
Epoch 17... Loss: 0.06402814388275146
Epoch 18... Loss: 0.20473375916481018
Epoch 19... Loss: 0.24928946793079376
Epoch 20... Loss: 0.09217169880867004
Epoch 21... Loss: 0.0982571691274643
Epoch 22... Loss: 0.11462420225143433
Epoch 23... Loss: 0.13200965523719788
Epoch 24... Loss: 0.14464005827903748
Epoch 25... Loss: 0.14718255400657654
Epoch 26... Loss: 0.07323227822780609
Epoch 27... Loss: 0.179623931